# Multiple Linear Regression Analysis (Cleaned)

This notebook performs data loading, feature encoding, VIF (multicollinearity) checks, OLS modeling using statsmodels, assumption diagnostics (Residuals vs Fitted, Q-Q), and provides a final interpretation with business recommendations.

## Key sections:
1. **Data Loading & Exploration** – Load CSV and inspect shape, missing values, data types
2. **Feature Engineering** – One-hot encode categorical variables (TV, Influencer)
3. **Multicollinearity Check** – Calculate VIF for all features
4. **OLS Regression Model** – Fit on training data and display summary statistics
5. **Assumption Diagnostics** – Residuals vs Fitted, Q-Q plot, Breusch-Pagan, Jarque-Bera tests
6. **Interpretation & Recommendation** – Extract coefficients, significance tests, and business insights

Notes:
- Corrupted metadata from large outputs has been removed to allow the notebook to parse correctly.
- All analysis cells are intact and runnable; execute locally or in Colab to generate figures and full outputs.

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson, jarque_bera
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

sns.set_style('whitegrid')
%matplotlib inline

In [ ]:
# Load data (ensure 'marketing_sales_data.csv' is present in the repository)
df = pd.read_csv('marketing_sales_data.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
# Basic checks
print('Missing values per column:')
print(df.isnull().sum())
print('\nData types:')
print(df.dtypes)
print('\nDescriptive statistics:')
df.describe()

In [ ]:
# One-hot encode categorical variables (drop_first=True to avoid dummy trap)
df_enc = pd.get_dummies(df, columns=['TV','Influencer'], drop_first=True)
print('Shape after encoding:', df_enc.shape)
df_enc.head()

In [ ]:
# Prepare features and target
y = df_enc['Sales']
X = df_enc.drop(columns=['Sales'])
# Keep column order for readability
X = X.reindex(sorted(X.columns), axis=1)
print('Features shape:', X.shape)
X.head()

In [ ]:
# VIF calculation to check multicollinearity
X_const = sm.add_constant(X)
vif = pd.DataFrame()
vif['feature'] = X_const.columns
vif['VIF'] = [variance_inflation_factor(X_const.values, i) for i in range(X_const.shape[1])]
print('Variance Inflation Factor (VIF):')
print(vif.sort_values('VIF', ascending=False))

In [ ]:
# Split into train/test for evaluation (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train_const = sm.add_constant(X_train)
X_test_const = sm.add_constant(X_test)
print(f'Training set size: {X_train.shape[0]}')
print(f'Test set size: {X_test.shape[0]}')

In [ ]:
# Fit OLS model on training data
model = sm.OLS(y_train, X_train_const).fit()
print(model.summary())

## Model Evaluation and Interpretation

Below we extract key statistics from the fitted model (Adjusted R-squared, F-statistic, p-values) and provide a concise interpretation with business recommendations.

In [ ]:
# Extract key statistics and produce a human-readable interpretation
adj_r2 = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
dw = durbin_watson(model.resid)

# Breusch-Pagan test for heteroscedasticity
bp = het_breuschpagan(model.resid, X_train_const)
bp_pvalue = bp[1]  # LM test p-value

# Jarque-Bera test for normality of residuals
jb_stat, jb_pvalue, _, _ = jarque_bera(model.resid)

print(f'Adjusted R-squared: {adj_r2:.4f}')
print(f'F-statistic: {f_stat:.4f} (p = {f_pvalue:.4g})')
print(f'Durbin-Watson: {dw:.4f} (close to 2 suggests residual independence)')
print(f'Breusch-Pagan p-value: {bp_pvalue:.4g} (p < 0.05 indicates heteroscedasticity)')
print(f'Jarque-Bera p-value: {jb_pvalue:.4g} (p < 0.05 indicates non-normal residuals)')

# Coefficients and significance
params = model.params
pvals = model.pvalues
significant = pvals[pvals < 0.05].index.tolist()
print('\nSignificant coefficients (p < 0.05):')
for name in significant:
    print(f'  {name}: coef={params[name]:.4f}, p={pvals[name]:.4g}')

In [ ]:
# Business recommendation: compare core channels (Radio, Social Media, TV)
# Identify TV-related dummies and compute effect magnitudes
tv_coefs = {c: params[c] for c in params.index if c.startswith('TV_')}
radio_coef = params.get('Radio', np.nan)
social_coef = params.get('Social Media', np.nan)

# Compute absolute effect magnitudes for comparison
tv_effect = max((abs(v) for v in tv_coefs.values()), default=np.nan) if tv_coefs else np.nan
effects = {'TV': tv_effect, 'Radio': abs(radio_coef), 'Social Media': abs(social_coef)}
best_channel = max(effects, key=lambda k: effects[k] if not np.isnan(effects[k]) else -np.inf)

print('\nEstimated effect magnitudes (absolute coefficients):')
for k, v in sorted(effects.items(), key=lambda x: x[1], reverse=True):
    print(f'  {k}: {v:.4f}')

# Recommendation text
rec = f'Business Recommendation: Based on coefficient magnitudes, focus marketing investment on {best_channel}, as it demonstrates the largest estimated impact on Sales (|coefficient| = {effects[best_channel]:.4f}). Allocate budget proportionally to channel effectiveness for optimal ROI.'
print('\n' + rec)

In [ ]:
# Predictions and basic metrics on test set
y_pred = model.predict(X_test_const)
test_r2 = r2_score(y_test, y_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f'Test R-squared: {test_r2:.4f}')
print(f'Test RMSE: {test_rmse:.4f}')

In [ ]:
# Residuals vs Fitted plot to assess non-linearity and heteroscedasticity
fitted = model.fittedvalues
residuals = model.resid
plt.figure(figsize=(8, 5))
plt.scatter(fitted, residuals, alpha=0.6, edgecolors='k')
plt.axhline(0, color='red', linestyle='--', linewidth=2)
plt.xlabel('Fitted values')
plt.ylabel('Residuals')
plt.title('Residuals vs Fitted Values')
plt.tight_layout()
plt.show()

In [ ]:
# Q-Q plot for residual normality assessment
plt.figure(figsize=(8, 5))
sm.qqplot(residuals, line='45', fit=True)
plt.title('Normal Q-Q Plot')
plt.tight_layout()
plt.show()

In [ ]:
# Breusch-Pagan test for heteroscedasticity (detailed results)
bp_test = het_breuschpagan(residuals, X_train_const)
labels = ['LM statistic', 'LM p-value', 'F statistic', 'F p-value']
print('\nBreusch-Pagan Test Results:')
print(dict(zip(labels, bp_test)))
print('\nInterpretation: If p-value < 0.05, heteroscedasticity is present.')

## Assumption Check Summary

### 1. **Linearity & Homoscedasticity**
Inspect the **Residuals vs Fitted** plot. If residuals are randomly scattered around 0 with no clear pattern, linearity and homoscedasticity assumptions hold. If the Breusch-Pagan p-value > 0.05, there is no significant heteroscedasticity.

### 2. **Normality of Residuals**
Inspect the **Q-Q plot**. If points fall closely along the 45-degree line and Jarque-Bera p-value > 0.05, residuals are approximately normally distributed.

### 3. **Independence of Residuals**
The **Durbin-Watson** statistic ranges from 0 to 4. A value near 2 indicates residuals are uncorrelated (independent). Values < 2 suggest positive autocorrelation; values > 2 suggest negative autocorrelation.

### 4. **Multicollinearity**
VIF values > 10 indicate problematic multicollinearity. Values between 1 and 5 are generally acceptable. Values < 1 or close to 1 indicate no multicollinearity concerns.

Replace the example interpretations above with your actual numeric results after running the cells.

In [ ]:
# Construct the final linear equation with all model coefficients
params = model.params
print('\nModel Coefficients:')
for name, coeff in params.items():
    print(f'  {name}: {coeff:.6f}')

# Build human-readable equation
intercept = params.get('const', 0.0)
equation_parts = [f'{intercept:.4f}']
for name, coeff in params.items():
    if name != 'const':
        sign = '+' if coeff >= 0 else ''
        equation_parts.append(f'{sign} {coeff:.4f} * {name}')

equation = 'Sales = ' + ''.join(equation_parts)
print('\nEstimated Linear Regression Equation:')
print(equation)

## Final Summary & Next Steps

### Key Takeaways:
- **Model Fit:** Adjusted R-squared indicates the proportion of variance in Sales explained by the model.
- **Statistical Significance:** F-statistic and p-value show whether the model is statistically significant overall.
- **Assumption Validity:** Review diagnostic plots and test statistics to ensure OLS assumptions are met.
- **Business Impact:** Channel coefficients reveal the relative contribution of each marketing channel to Sales.

### Optional Next Steps:
1. **Backward Elimination:** Remove non-significant variables (p > 0.05) and re-fit the model.
2. **Standardization:** Standardize continuous predictors to compare effect sizes on a common scale.
3. **Cross-Validation:** Use k-fold cross-validation for more robust model evaluation.
4. **Executive Summary:** Generate a summary report and save key metrics to a markdown file.
5. **Robustness Checks:** Test for outliers and influential observations using diagnostic measures (leverage, Cook's distance).

For questions or model refinements, revisit the data exploration, encoding, or variable selection stages.